## Prepare questions from ADNi for testing GRPO model

Created by: Sahana Kowshik

Date: 11/01/2025

In [1]:
import pandas as pd
import json
import random

In [2]:
directory_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/adni"
test = pd.read_csv(f"{directory_path}/training_data/testing_data_grpo/test_summary.csv")

In [3]:
test['COHORT'] = "ADNI"

In [4]:
print(test.iloc[0]['visit_summary'])

The subject is a 70.6-year-old White, female participant who is not Hispanic or Latino and speaks English. She was born in July 1947 and has completed 16 years of education. Her family history reveals that both her biological parents have deceased, with the mother passing away at age 69 and the father at an unspecified age. Both parents had dementia, but neither had Alzheimer’s disease; symptoms began at age 65 for the mother and at age 67 for the father. In terms of health history, the participant has a variety of chronic conditions including hearing loss, hypothyroidism, hypertension, hyperlipidemia, GERD, dry eyes, rhinorrhea, rosacea, spinal stenosis, and depression. Additionally, she has a history of surgeries related to the removal of a pre-melanoma lesion, polyps, and a partial thyroidectomy due to a goiter. Codeine use has been associated with mood swings. Her current physical condition includes a weight of 86.8 kilograms, a systolic blood pressure of 140 mmHg, a diastolic bloo

In [5]:
len(test)

1392

In [6]:
test.head()

,RID,DIAGNOSIS,PTGENDER,PTDOBMM,PTDOBYY,PTEDUCAT,PTTLANG,PTETHCAT,PTRACCAT,CENTILOIDS,...,PHC_pTau,AT_class,AGE_MRI,AMYLOID_PET_POSITIVE,brain_findings_summary,patient_summary,diag_summary,visit_summary_prompt,visit_summary,COHORT
0,6234,1.0,2.0,7,1947,16.0,1.0,2.0,5.0,19.0,...,NaN,NaN,70.639288,0,Medial Temporal region has mild atrophy,"{""Subject Demographics"": {""1. Participant Gend...","{""Which best describes the participant's curre...",<|im_start|>user\nYou will receive patient dat...,"The subject is a 70.6-year-old White, female p...",ADNI
1,2308,2.0,1.0,4,1936,15.0,1.0,2.0,5.0,-6.0,...,NaN,A-T-,74.880219,0,No significant findings. Brain MRI appears age...,"{""Subject Demographics"": {""1. Participant Gend...","{""Which best describes the participant's curre...",<|im_start|>user\nYou will receive patient dat...,The subject is a 74.88-year-old male participa...,ADNI
2,7042,1.0,2.0,6,1960,16.0,1.0,2.0,4.0,-4.0,...,NaN,NaN,61.727584,0,Medial Parietal region has mild atrophy; Infer...,"{""Subject Demographics"": {""1. Participant Gend...","{""Which best describes the participant's curre...",<|im_start|>user\nYou will receive patient dat...,The subject is a 61.73-year-old Black or Afric...,ADNI
3,6147,1.0,1.0,6,1948,16.0,1.0,2.0,5.0,24.0,...,NaN,NaN,69.702943,1,Hippocampus region has mild atrophy,"{""Subject Demographics"": {""1. Participant Gend...","{""Which best describes the participant's curre...",<|im_start|>user\nYou will receive patient dat...,The subject is a 69.7-year-old male born in Ju...,ADNI
4,4444,2.0,1.0,6,1934,18.0,1.0,2.0,5.0,30.0,...,NaN,NaN,77.705681,1,No significant findings. Brain MRI appears age...,"{""Subject Demographics"": {""1. Participant Gend...","{""Which best describes the participant's curre...",<|im_start|>user\nYou will receive patient dat...,The patient is a 77.7-year-old male born in Ju...,ADNI


In [7]:
test['DIAGNOSIS'].isna().sum()

0

In [8]:
test['AMYLOID_PET_POSITIVE'].isna().sum()

0

In [9]:
test['DIAGNOSIS'].value_counts()

DIAGNOSIS
2.0    603
1.0    555
3.0    234
Name: count, dtype: int64

In [10]:
columns = ['ID', 'patient_summary', 'diag_summary', 'visit_summary', 'question', 'options', 'ground_truth', 'ground_truth_text', 'COHORT']

# Add cognitive status question


In [11]:
import random, string

# Question variants for cognitive status identification
cog_question_variants = [
    "Using the information provided, determine the patient's cognitive status.",
    "Identify the patient's cognitive status based on the given information.",
    "Based on the provided clinical details, classify the patient's cognitive status.",
    "From the information available, determine the correct cognitive status for the patient.",
    "Analyze the patient's information and identify their cognitive status."
]

# Original answer texts
COG_OPTION_TEXTS = [
    "Normal Cognition (NC)",
    "Mild Cognitive Impairment (MCI)",
    "Dementia (DE)"
]

# Mapping from NACCUDSD to the correct label text
COG_ANSWER_MAP = {
    1: COG_OPTION_TEXTS[0],
    2: COG_OPTION_TEXTS[1],
    3: COG_OPTION_TEXTS[2],
}

def add_cog_question(row, *, seed=None):
    rng = random.Random(seed if seed is not None else row.name)

    # 1️⃣ Random question phrasing
    row['question'] = rng.choice(cog_question_variants)

    # 2️⃣ Shuffle the answer options
    shuffled = COG_OPTION_TEXTS.copy()
    rng.shuffle(shuffled)
    letters = list(string.ascii_uppercase[:len(shuffled)])
    row['options'] = "\n".join(f"{ltr}. {opt}" for ltr, opt in zip(letters, shuffled))

    # 3️⃣ Determine correct answer letter
    correct_text = COG_ANSWER_MAP.get(row['DIAGNOSIS'], None)
    if correct_text in shuffled:
        row['ground_truth'] = letters[shuffled.index(correct_text)]
    else:
        row['ground_truth'] = None  # fallback for unrecognized codes
    
    row['ground_truth_text'] = correct_text

    return row


In [12]:
# Apply the function to the MCI training set
test_cog = test.apply(add_cog_question, axis=1)

In [13]:
test_cog['ground_truth_text'].value_counts()

ground_truth_text
Mild Cognitive Impairment (MCI)    603
Normal Cognition (NC)              555
Dementia (DE)                      234
Name: count, dtype: int64

In [14]:
test_cog["ID"] = test_cog["RID"]
test_cog = test_cog[columns]
print(len(test_cog))
test_cog = test_cog.dropna(how='any').reset_index(drop=True)
print(len(test_cog))

1392
1392


/scratch/ipykernel_225100/2269045022.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_cog["ID"] = test_cog["RID"]


In [15]:
test_cog['Q_TYPE'] = "Cognitive status"

In [16]:
test_cog.head()

,ID,patient_summary,diag_summary,visit_summary,question,options,ground_truth,ground_truth_text,COHORT,Q_TYPE
0,6234,"{""Subject Demographics"": {""1. Participant Gend...","{""Which best describes the participant's curre...","The subject is a 70.6-year-old White, female p...","From the information available, determine the ...",A. Dementia (DE)\nB. Normal Cognition (NC)\nC....,B,Normal Cognition (NC),ADNI,Cognitive status
1,2308,"{""Subject Demographics"": {""1. Participant Gend...","{""Which best describes the participant's curre...",The subject is a 74.88-year-old male participa...,Identify the patient's cognitive status based ...,A. Mild Cognitive Impairment (MCI)\nB. Normal ...,A,Mild Cognitive Impairment (MCI),ADNI,Cognitive status
2,7042,"{""Subject Demographics"": {""1. Participant Gend...","{""Which best describes the participant's curre...",The subject is a 61.73-year-old Black or Afric...,"Using the information provided, determine the ...",A. Mild Cognitive Impairment (MCI)\nB. Dementi...,C,Normal Cognition (NC),ADNI,Cognitive status
3,6147,"{""Subject Demographics"": {""1. Participant Gend...","{""Which best describes the participant's curre...",The subject is a 69.7-year-old male born in Ju...,Identify the patient's cognitive status based ...,A. Mild Cognitive Impairment (MCI)\nB. Normal ...,B,Normal Cognition (NC),ADNI,Cognitive status
4,4444,"{""Subject Demographics"": {""1. Participant Gend...","{""Which best describes the participant's curre...",The patient is a 77.7-year-old male born in Ju...,Identify the patient's cognitive status based ...,A. Dementia (DE)\nB. Normal Cognition (NC)\nC....,C,Mild Cognitive Impairment (MCI),ADNI,Cognitive status


In [17]:
test_cog.to_csv(f"{directory_path}/training_data/testing_data_grpo/with_summary/test_cog.csv", index=False)

# Amyloid

In [18]:
test['AMYLOID_PET_POSITIVE'].value_counts()

AMYLOID_PET_POSITIVE
0    715
1    677
Name: count, dtype: int64

In [19]:
test_pet = test[~test['AMYLOID_PET_POSITIVE'].isna()].copy().reset_index(drop=True)

In [20]:
json.loads(test_pet.iloc[1]['patient_summary'])

{'Subject Demographics': {'1. Participant Gender': 'Male',
  '2a. Participant Month of Birth': 4,
  '2b. Participant Year of Birth': 1936,
  '5. Participant Education': 15.0,
  '9. Language to be used for testing the Participant': 'English',
  '12. Ethnic Category': 'Not Hispanic or Latino',
  '13. Racial Categories': 'White',
  'Subject age at MRI acquisition': 74.88021902806297},
 'Subject Health History': {'Initial Health Assessment - Description of condition': 'Rhinoplasty, (Surgery: Yes), (Surgery Date: 1964-07-01); High Cholesterol, (Surgery: No), (Present in 3 months prior to screening: Yes), (Chronicity: Persistent), (Severity: Moderate), (Onset Date: 2010-07-02), (Ongoing: Yes); Hypertension, (Surgery: No), (Present in 3 months prior to screening: Yes), (Chronicity: Persistent), (Severity: Moderate), (Onset Date: 2006-07-02), (Ongoing: Yes); Chronic Fatigue Syndrome, (Surgery: No), (Present in 3 months prior to screening: Yes), (Chronicity: Persistent), (Severity: Moderate), (

In [21]:
import random, string, json

# Keys related to amyloid biomarkers that should be removed from the patient summary
# pet_keys_remove = [
#     'Abnormally elevated amyloid on PET',
#     'Abnormally low amyloid in CSF',
#     'FDG-PET pattern of AD',
#     'Tau PET evidence for AD',
#     'Abnormally elevated CSF Tau or pTau'
# ]

pet_question_variants = [
    "Is the patient likely to have elevated amyloid pathology in the brain given the provided information?",
    "Does the patient likely have abnormal amyloid accumulation in the brain based on the clinical data?",
    "Based on the available information, is the patient likely amyloid positive in the brain?",
    "Does the evidence suggest the presence of elevated amyloid burden in the brain in this patient?",
    "Is there a high likelihood of underlying amyloid pathology in the brain based on the patient's profile?"
]


# Raw answer texts
PET_OPTION_TEXTS = ["Yes", "No"]

# Map AMYLPET value to answer text
PET_ANSWER_MAP = {
    1: "Yes",  # Amyloid positive
    0: "No",   # Amyloid negative
}

def add_pet_question(row, *, seed=None):
    rng = random.Random(seed if seed is not None else row.name)

    # 1️⃣ Choose a question variant randomly
    row['question'] = rng.choice(pet_question_variants)

    # 2️⃣ Remove sensitive fields from the summary
    # json_file = json.loads(row['patient_summary'])
    # for key in pet_keys_remove:
    #     if key in json_file.get('Imaging evidence', {}):
    #         json_file['Imaging evidence'].pop(key)
    #     if key in json_file.get('CSF evidence', {}):
    #         json_file['CSF evidence'].pop(key)
    # row['patient_summary'] = json.dumps(json_file, indent=4)

    # 3️⃣ Shuffle the answer options
    shuffled = PET_OPTION_TEXTS.copy()
    rng.shuffle(shuffled)
    letters = list(string.ascii_uppercase[:len(shuffled)])
    row["options"] = "\n".join(f"{ltr}. {opt}" for ltr, opt in zip(letters, shuffled))

    # 4️⃣ Determine correct letter based on AMYLPET value
    correct_text = PET_ANSWER_MAP[row['AMYLOID_PET_POSITIVE']]
    correct_letter = letters[shuffled.index(correct_text)]
    row["ground_truth"] = correct_letter
    
    row['ground_truth_text'] = correct_text

    return row


In [22]:
# Apply the function to the MCI training set
test_pet = test.apply(add_pet_question, axis=1)

In [23]:
test_pet['ground_truth_text'].value_counts()

ground_truth_text
No     715
Yes    677
Name: count, dtype: int64

In [24]:
test_pet["ID"] = test_pet["RID"]
test_pet = test_pet[columns]
print(len(test_pet))
test_pet = test_pet.dropna(how='any').reset_index(drop=True)
print(len(test_pet))

1392
1392


/scratch/ipykernel_225100/2769316785.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_pet["ID"] = test_pet["RID"]


In [25]:
test_pet['Q_TYPE'] = "Amyloid PET"

In [26]:
test_pet.head()

,ID,patient_summary,diag_summary,visit_summary,question,options,ground_truth,ground_truth_text,COHORT,Q_TYPE
0,6234,"{""Subject Demographics"": {""1. Participant Gend...","{""Which best describes the participant's curre...","The subject is a 70.6-year-old White, female p...",Does the evidence suggest the presence of elev...,A. Yes\nB. No,B,No,ADNI,Amyloid PET
1,2308,"{""Subject Demographics"": {""1. Participant Gend...","{""Which best describes the participant's curre...",The subject is a 74.88-year-old male participa...,Does the patient likely have abnormal amyloid ...,A. No\nB. Yes,A,No,ADNI,Amyloid PET
2,7042,"{""Subject Demographics"": {""1. Participant Gend...","{""Which best describes the participant's curre...",The subject is a 61.73-year-old Black or Afric...,Is the patient likely to have elevated amyloid...,A. No\nB. Yes,A,No,ADNI,Amyloid PET
3,6147,"{""Subject Demographics"": {""1. Participant Gend...","{""Which best describes the participant's curre...",The subject is a 69.7-year-old male born in Ju...,Does the patient likely have abnormal amyloid ...,A. No\nB. Yes,B,Yes,ADNI,Amyloid PET
4,4444,"{""Subject Demographics"": {""1. Participant Gend...","{""Which best describes the participant's curre...",The patient is a 77.7-year-old male born in Ju...,Does the patient likely have abnormal amyloid ...,A. Yes\nB. No,A,Yes,ADNI,Amyloid PET


In [27]:
test_pet.to_csv(f"{directory_path}/training_data/testing_data_grpo/with_summary/test_pet.csv", index=False)

# Add primary etiology question

In [28]:
test['DIAGNOSIS'].value_counts()

DIAGNOSIS
2.0    603
1.0    555
3.0    234
Name: count, dtype: int64

In [29]:
def add_ad_label(row):
    if (row['DIAGNOSIS'] in [2.0, 3.0]) & (row['AMYLOID_PET_POSITIVE'] == 1):
        return 'AD'
    elif row['DIAGNOSIS'] == 1.0:
        return 'NC'

test['AD_LABEL'] = test.apply(add_ad_label, axis=1)
test['AD_LABEL'].value_counts()


AD_LABEL
NC    555
AD    504
Name: count, dtype: int64

In [30]:
test[test['AD_LABEL'].isna()]['AMYLOID_PET_POSITIVE'].value_counts()

AMYLOID_PET_POSITIVE
0    333
Name: count, dtype: int64

In [31]:
import random, string

etiology_question_variants = [
    "Identify the primary etiologic diagnosis of the patient using the information provided, if applicable.",
    "Identify the primary etiology of the patient's cognitive impairment using the information provided, if applicable.",
    "Based on the clinical details, determine the main cause of the patient's cognitive impairment, if applicable.",
    "Using the information provided, identify the primary cause of the patient's cognitive decline, if applicable.",
    "Determine the primary etiology underlying the patient's cognitive impairment based on the provided information, if applicable."
]

# Raw etiology answer texts in original order
ETIOLOGY_OPTION_TEXTS = [
    "Alzheimer's disease (AD)",
    "Lewy body disease (LBD)",
    "Frontotemporal lobar degeneration and its variants, including primary progressive aphasia, corticobasal degeneration and progressive supranuclear palsy, and with or without amyotrophic lateral sclerosis (FTLD)",
    "Vascular brain injury or vascular dementia including stroke (VD)",
    "Systemic and environmental factors including infectious diseases (HIV included), metabolic, substance abuse / alcohol, medications, systemic disease and delirium (SEF)",
    "Psychiatric conditions including schizophrenia, depression, bipolar disorder, anxiety and posttraumatic stress disorder (PSY)",
    "Other (Multiple system atrophy, Essential tremor, Down syndrome, Huntington's disease, Prion disease, Traumatic brain injury, Normal-pressure hydrocephalus, Epilepsy, CNS neoplasm, etc)",
    "Not applicable (no cognitive impairment)",
]

# Mapping of ETPR code to correct answer text
ETPR_ANSWER_MAP = {
    'AD': ETIOLOGY_OPTION_TEXTS[0],
    'NC': ETIOLOGY_OPTION_TEXTS[-1],
}

def add_etpr_question(row, *, seed=None):
    rng = random.Random(seed if seed is not None else row.name)

    # 1️⃣ Randomly select a question variant
    row['question'] = rng.choice(etiology_question_variants)

    # 2️⃣ Shuffle options and assign fresh letters
    shuffled = ETIOLOGY_OPTION_TEXTS.copy()
    rng.shuffle(shuffled)
    letters = list(string.ascii_uppercase[:len(shuffled)])
    row['options'] = "\n".join(f"{ltr}. {opt}" for ltr, opt in zip(letters, shuffled))

    # 3️⃣ Get correct text from ETPR code (default to "Not applicable" if unknown)
    correct_text = ETPR_ANSWER_MAP.get(row['AD_LABEL'])

    if correct_text is None:
        raise ValueError(f"Invalid ETPR value: {row['AD_LABEL']}")


    # 4️⃣ Get ground truth letter from shuffled list
    if correct_text in shuffled:
        row['ground_truth'] = letters[shuffled.index(correct_text)]
    else:
        row['ground_truth'] = None  # shouldn't happen unless text mismatch
        
    row['ground_truth_text'] = correct_text

    return row


In [32]:
# Apply the function to the MCI training set
test_etpr = test[~test['AD_LABEL'].isna()]
test_etpr = test_etpr.apply(add_etpr_question, axis=1)

In [33]:
test_etpr['ground_truth_text'].value_counts()

ground_truth_text
Not applicable (no cognitive impairment)    555
Alzheimer's disease (AD)                    504
Name: count, dtype: int64

In [34]:
test_etpr["ID"] = test_etpr["RID"]
test_etpr = test_etpr[columns]
print(len(test_etpr))
test_etpr = test_etpr.dropna(how='any').reset_index(drop=True)
print(len(test_etpr))

1059
1059


/scratch/ipykernel_225100/3397397685.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test_etpr["ID"] = test_etpr["RID"]


In [35]:
test_etpr['Q_TYPE'] = "Primary etiology"

In [36]:
test_etpr.head()

,ID,patient_summary,diag_summary,visit_summary,question,options,ground_truth,ground_truth_text,COHORT,Q_TYPE
0,6234,"{""Subject Demographics"": {""1. Participant Gend...","{""Which best describes the participant's curre...","The subject is a 70.6-year-old White, female p...","Using the information provided, identify the p...",A. Not applicable (no cognitive impairment)\nB...,A,Not applicable (no cognitive impairment),ADNI,Primary etiology
1,7042,"{""Subject Demographics"": {""1. Participant Gend...","{""Which best describes the participant's curre...",The subject is a 61.73-year-old Black or Afric...,Identify the primary etiologic diagnosis of th...,A. Vascular brain injury or vascular dementia ...,E,Not applicable (no cognitive impairment),ADNI,Primary etiology
2,6147,"{""Subject Demographics"": {""1. Participant Gend...","{""Which best describes the participant's curre...",The subject is a 69.7-year-old male born in Ju...,Identify the primary etiology of the patient's...,A. Lewy body disease (LBD)\nB. Psychiatric con...,G,Not applicable (no cognitive impairment),ADNI,Primary etiology
3,4444,"{""Subject Demographics"": {""1. Participant Gend...","{""Which best describes the participant's curre...",The patient is a 77.7-year-old male born in Ju...,Identify the primary etiology of the patient's...,A. Lewy body disease (LBD)\nB. Frontotemporal ...,G,Alzheimer's disease (AD),ADNI,Primary etiology
4,6488,"{""Subject Demographics"": {""1. Participant Gend...","{""Which best describes the participant's curre...","The subject is a 76.6-year-old White, non-Hisp...",Determine the primary etiology underlying the ...,"A. Other (Multiple system atrophy, Essential t...",E,Not applicable (no cognitive impairment),ADNI,Primary etiology


In [37]:
test_etpr.to_csv(f"{directory_path}/training_data/testing_data_grpo/with_summary/test_etpr.csv", index=False)